In [2]:
%cd ..

/home/oleg/audio-llm-yandex-camp


In [ ]:
from src.asr_eval.align.data import MatchesList, Token
from src.asr_eval.align.parsing import split_text_into_tokens
from src.asr_eval.align.recursive import align
from src.asr_eval.utils.formatting import Formatting, FormattingSpan, apply_ansi_formatting
from termcolor import colored
import pandas as pd

truth = split_text_into_tokens('раз два три четыре')
preds = {
    'truth': truth,
    'model1': split_text_into_tokens('раз один один два три четыре ы'),
    'model2': split_text_into_tokens('а раз один два три четыре'),
    'model3': split_text_into_tokens('раз два три четыре пять'),
}

def align_into_table(truth: list[Token], pred: MatchesList) -> list[Token | None | list[Token]]:
    '''
    Имея truth длиной N, возвращает список длиной N + 1
    Нечетные элементы списка соответствуют позициям в truth
    Четные элементы соответствуют промежуткам между ними.

    Например, для len(truth) == 2:

    элемент 0 выходного массива: токены из pred перед 1-й позицией в truth
    элемент 1 выходного массива: токен из pred для 1-й позиции в truth
    элемент 2 выходного массива: токены из pred между 1-й и 2-й позицией в truth
    элемент 3 выходного массива: токен из pred для 2-й позиции в truth
    элемент 4 выходного массива: токены из pred после 2-й позицией в truth
    '''
    result: list[Token | None | list[Token]] = [None] * (2 * len(truth) + 1)

    true_index = 0  # если 0, значит мы на 0м токене из truth или перед ним

    def true_index_to_pred_index(i: int) -> int:
        return 2 * i + 1

    for match in pred.matches:
        if match.true is not None:
            # соответствует какому-то токену из truth
            table_index = true_index_to_pred_index(true_index)
            result[table_index] = match.pred
            true_index += 1
        else:
            # не соответствует никакому токену из truth (т. е. вставка)
            table_index = true_index_to_pred_index(true_index) - 1
            if result[table_index] is None:
                result[table_index] = []
            result[table_index].append(match.pred) # type: ignore

    return result

rows = {}
for model_name, pred_tokens in preds.items():
    row = align_into_table(truth, align(truth, pred_tokens)) # type: ignore
    rows[model_name] = row

table = pd.DataFrame(rows).T

def cell_to_str(cell: Token | None | list[Token]) -> str:
    if cell is None:
        return ''
    elif isinstance(cell, list):
        return ' '.join(str(t.value) for t in cell) 
    else:
        return str(cell.value)

table = table.map(cell_to_str) # type: ignore

table



,0,1,2,3,4,5,6,7,8
truth,,раз,,два,,три,,четыре,
model1,,раз,один один,два,,три,,четыре,ы
model2,а,раз,один,два,,три,,четыре,
model3,,раз,,два,,три,,четыре,пять


In [47]:
table = pd.DataFrame({
    col_name: col for col_name, col in table.items() # type: ignore
    if max(len(x) for x in col.values) > 0 # type: ignore
})

print('\n'.join(str(table).splitlines()[1:]))

truth      раз             два  три  четыре      
model1     раз  один один  два  три  четыре     ы
model2  а  раз       один  два  три  четыре      
model3     раз             два  три  четыре  пять


In [48]:
1

1